In [ ]:
%pip install "transformers==5.17.0" "librosa==1.0.0" \
    "demucs==4.1.0" "jiwer==4.0.0" "soundfile==0.14.0" \
    --quiet


# Solution — Module M5: เทรนโมเดลฟังเสียงจากศูนย์ (KWS) + วิเคราะห์ ASR

เฉลยครบทุกข้อของ `exercise.ipynb` (ผู้สอนเปิดเผยหลังจบแล็บ) — ใช้ค่า scaffold เดียวกัน (`MAX_PER_WORD = 600`, `UNKNOWN_N = 800`, `EPOCHS = 3`) เพื่อให้ตัวเลขเทียบกับที่ผู้เรียนได้ตรงกัน ค่าที่พิมพ์ในคำตอบเป็นค่าจากรอบ seed 42 — รอบของคุณอาจต่างไปเล็กน้อย


## 0. เตรียมข้อมูล + เทรน (ให้มาแล้ว — รันผ่าน)

โค้ดชุดนี้คือส่วนสำคัญของ book ย่อมาไว้ด้วยกัน: ดาวน์โหลด Speech Commands v2 → เตรียม 12 class (10 คำสั่ง + `_unknown_` + `_silence_`) → แปลง log-mel → เทรน CNN จากศูนย์ 3 epochs (จะได้ `model`, `cm`, `SHORT`, `test_pairs`, `y_pred` ไว้ใช้ทำข้อ 1–3)


In [ ]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np  # noqa: E402
import torch  # noqa: E402

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ไม่มี GPU — ไปตั้งค่า Settings > Accelerator: GPU T4")


In [ ]:
import tarfile  # noqa: E402
import urllib.request  # noqa: E402
from pathlib import Path  # noqa: E402

SC_URL = (
    "http://download.tensorflow.org/data/"
    "speech_commands_v0.02.tar.gz"
)
SC_ROOT = Path("/kaggle/working/speech_commands_v2")
ARCHIVE = "/kaggle/working/speech_commands_v0.02.tar.gz"

if not (SC_ROOT / "testing_list.txt").exists():
    if not Path(ARCHIVE).exists():
        print("กำลังดาวน์โหลด ~2.4 GB (2-5 นาที) ...")
        urllib.request.urlretrieve(SC_URL, ARCHIVE)
    print("กำลังแตกไฟล์ (1-2 นาที) ...")
    with tarfile.open(ARCHIVE) as tar:
        tar.extractall(SC_ROOT, filter="data")
    Path(ARCHIVE).unlink()
print("dataset พร้อมที่:", SC_ROOT)
print("จำนวนไฟล์ .wav:", len(list(SC_ROOT.glob("*/*.wav"))))


In [ ]:
import random  # noqa: E402

CMD_WORDS = [
    "down", "go", "left", "no", "off",
    "on", "right", "stop", "up", "yes",
]
LABELS = CMD_WORDS + ["_unknown_", "_silence_"]
MAX_PER_WORD = 600
UNKNOWN_N = 800

val_set = set((SC_ROOT / "validation_list.txt").read_text().split())
test_set = set((SC_ROOT / "testing_list.txt").read_text().split())

def split_of(path):
    rel = f"{path.parent.name}/{path.name}"
    if rel in test_set:
        return "test"
    if rel in val_set:
        return "val"
    return "train"

buckets = {"train": [], "val": [], "test": []}
for path in sorted(SC_ROOT.glob("*/*.wav")):
    word = path.parent.name
    if word == "_background_noise_":
        continue
    label = word if word in CMD_WORDS else "_unknown_"
    buckets[split_of(path)].append((path, label))

rng = random.Random(SEED)
by_word = {}
for path, label in buckets["train"]:
    by_word.setdefault(label, []).append((path, label))

train_pairs = []
for word in CMD_WORDS:
    train_pairs.extend(sorted(by_word.get(word, []))[:MAX_PER_WORD])
unknown_train = sorted(by_word.get("_unknown_", []))
rng.shuffle(unknown_train)
train_pairs.extend(unknown_train[:UNKNOWN_N])
val_pairs = buckets["val"]
test_pairs = buckets["test"]

splits = [
    ("train", train_pairs),
    ("val", val_pairs),
    ("test", test_pairs),
]
for name, pairs in splits:
    print(f"{name}: {len(pairs)} ตัวอย่าง")


In [ ]:
import matplotlib.pyplot as plt  # noqa: E402, F401
import soundfile as sf  # noqa: E402
import torchaudio  # noqa: E402

SAMPLES = 16000

mel_fn = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLES, n_fft=1024, hop_length=256, n_mels=64,
).to(DEVICE)
to_db = torchaudio.transforms.AmplitudeToDB().to(DEVICE)

def load_wave(path):
    wave, sr = sf.read(str(path), dtype="float32")
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    wave = wave[:SAMPLES]
    if len(wave) < SAMPLES:
        wave = np.pad(wave, (0, SAMPLES - len(wave)))
    return wave


In [ ]:
def make_silence(n, rng_local):
    """สร้างคลิป _silence_ จากเสียงห้องจริง (สุ่มช่วง+ความดัง)."""
    out = []
    for _ in range(n):
        path = noise_files[rng_local.randrange(len(noise_files))]
        full, _ = sf.read(str(path), dtype="float32")
        if full.ndim > 1:
            full = full.mean(axis=1)
        start = rng_local.randrange(0, len(full) - SAMPLES - 1)
        clip = full[start:start + SAMPLES].copy()
        out.append(clip * rng_local.uniform(0.0, 1.0))
    return out

def featurize(pairs, silence_waves):
    waves = [load_wave(path) for path, _ in pairs]
    labels = [LABELS.index(w) for _, w in pairs]
    waves += silence_waves
    labels += [LABELS.index("_silence_")] * len(silence_waves)
    feats = []
    with torch.no_grad():
        for i in range(0, len(waves), 512):
            x = torch.from_numpy(np.stack(waves[i:i + 512]))
            feats.append(to_db(mel_fn(x.to(DEVICE))).cpu())
    x_all = torch.cat(feats).unsqueeze(1)
    y_all = torch.tensor(labels)
    return x_all, y_all

noise_files = sorted(
    (SC_ROOT / "_background_noise_").glob("*.wav")
)
print("เตรียม train ...")
X_train, y_train = featurize(train_pairs, make_silence(200, rng))
print("เตรียม val ...")
X_val, y_val = featurize(val_pairs, make_silence(60, rng))
print("เตรียม test ...")
X_test, y_test = featurize(test_pairs, make_silence(60, rng))
print("train:", tuple(X_train.shape), "| val:",
      tuple(X_val.shape), "| test:", tuple(X_test.shape))


In [ ]:
class KwsNet(torch.nn.Module):
    """CNN เล็ก ๆ สำหรับ 12-class keyword spotting."""

    def __init__(self, n_classes):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 16, 3, padding=1),
            torch.nn.BatchNorm2d(16),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(16, 32, 3, padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(32, 64, 3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(64 * 8 * 7, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


In [ ]:
from torch.utils.data import (  # noqa: E402
    DataLoader,
    TensorDataset,
)

BATCH = 128
EPOCHS = 3

@torch.no_grad()
def accuracy(net, X, y, batch=512):
    net.eval()
    correct = 0
    for i in range(0, len(X), batch):
        logits = net(X[i:i + batch].to(DEVICE))
        pred = logits.argmax(dim=-1).cpu()
        correct += int((pred == y[i:i + batch]).sum())
    return correct / len(X)

def train_model(net, epochs=EPOCHS, lr=1e-3, batch_size=BATCH):
    """เทรน net บน X_train — คืน (loss, acc train, acc val) ราย epoch."""
    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
    )
    history = []
    for epoch in range(1, epochs + 1):
        net.train()
        total_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(yb)
        train_loss = total_loss / len(y_train)
        tr = accuracy(net, X_train, y_train)
        va = accuracy(net, X_val, y_val)
        history.append((train_loss, tr, va))
        print(
            f"epoch {epoch}/{epochs} loss={train_loss:.3f} "
            f"acc train={tr:.3f} val={va:.3f}"
        )
    return history

model = KwsNet(n_classes=len(LABELS)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"พารามิเตอร์ทั้งหมด: {n_params:,}")
history = train_model(model)


ต่อ: วัดผลบน test set — จะได้ `y_pred`, `cm`, `SHORT`, `label_map` (ชื่อย่อ -> ชื่อเต็มใน LABELS) และ `test_acc` ไว้ใช้ข้อ 1–3:


In [ ]:
from sklearn.metrics import confusion_matrix  # noqa: E402

@torch.no_grad()
def predict_all(net, X, batch=512):
    net.eval()
    preds = []
    for i in range(0, len(X), batch):
        logits = net(X[i:i + batch].to(DEVICE))
        preds.append(logits.argmax(dim=-1).cpu())
    return torch.cat(preds).numpy()

y_pred = predict_all(model, X_test)
correct = int((y_pred == y_test.numpy()).sum())
test_acc = correct / len(y_test)
print(f"test accuracy: {test_acc:.3f} ({correct}/{len(y_test)})")

SHORT = [
    "down", "go", "left", "no", "off",
    "on", "right", "stop", "up", "yes", "unk", "sil",
]
label_map = dict(zip(SHORT, LABELS))
cm = confusion_matrix(y_test.numpy(), y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(SHORT)))
ax.set_xticklabels(SHORT, rotation=45)
ax.set_yticks(range(len(SHORT)))
ax.set_yticklabels(SHORT)
ax.set_xlabel("โมเดลทำนาย")
ax.set_ylabel("คำตอบจริง")
fig.colorbar(im)
plt.tight_layout()
plt.show()

pairs = []
for i in range(len(SHORT)):
    for j in range(len(SHORT)):
        if i != j and cm[i, j] > 0:
            pairs.append((int(cm[i, j]), SHORT[i], SHORT[j]))
pairs.sort(reverse=True)
print("คู่ที่สับสนมากที่สุด 8 อันดับ (จริง -> ทำนาย):")
for n, a, b in pairs[:8]:
    print(f"  {a:>6} -> {b:<6} : {n} ครั้ง")


## ข้อ 1 — โมเดลเล็กลง 4 เท่า แม่นลดเท่าไร?

เฉลย: `KwsNetSmall` ตามสเปก (conv 8/16/32 + head 64) — input ของ linear ชั้นแรกคือ 32 × 8 × 7 = 1,792


In [ ]:
class KwsNetSmall(torch.nn.Module):
    """KwsNet ฉบับย่อ: channel 8/16/32 + head 64 (~0.12M params)."""

    def __init__(self, n_classes):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 8, 3, padding=1),
            torch.nn.BatchNorm2d(8),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(8, 16, 3, padding=1),
            torch.nn.BatchNorm2d(16),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(16, 32, 3, padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(32 * 8 * 7, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

small = KwsNetSmall(n_classes=len(LABELS)).to(DEVICE)
n_small = sum(p.numel() for p in small.parameters())
print(f"พารามิเตอร์: เต็ม {n_params:,} | เล็ก {n_small:,} "
      f"(ลด {1 - n_small / n_params:.0%})")

history_small = train_model(small)
print(f"val accuracy: เต็ม {history[-1][2]:.3f} | "
      f"เล็ก {history_small[-1][2]:.3f}")
small_pred = predict_all(small, X_test)
small_acc = float((small_pred == y_test.numpy()).mean())
print(f"test accuracy: เต็ม {test_acc:.3f} | เล็ก {small_acc:.3f}")


**คำตอบข้อ 1:** โมเดลเล็กมีพารามิเตอร์ ~0.12M vs ~0.50M (ลดราว 4 เท่า) จากรอบ seed 42 val accuracy ต่างกันประมาณ 1–4 หน่วยเปอร์เซ็นต์ (เช่น ~0.93 vs ~0.95) — แปลว่าโจทย์ "แยก 12 เสียงพูด 1 วินาที" **ไม่ต้องใช้โมเดลใหญ่** — โครงสร้างที่เหมาะสมสำคัญกว่าขนาด จุดที่แม่นลดมักเป็น `_unknown_` และคู่ที่เสียงใกล้กัน (go/no, yes/right) เพราะ capacity น้อยลงจับรายละเอียดความถี่ได้น้อยลง ถ้าจะเอาไปใช้จริงบน ไมโครคอนโทรลเลอร์ โมเดลเล็กนี่แหละที่คุ้ม


## ข้อ 2 — คู่สับสนที่แย่ที่สุด: ฟังดูว่า "เพราะอะไร"


In [ ]:
from IPython.display import Audio, display  # noqa: E402

n_wrong, a, b = pairs[0]
ia = LABELS.index(label_map[a])
ib = LABELS.index(label_map[b])
wrong_idx = [
    i for i in range(len(test_pairs))
    if y_test.numpy()[i] == ia and y_pred[i] == ib
]
print(f"คู่แย่สุด: {a} -> {b} ({n_wrong} ครั้ง) — "
      f"ตัวอย่างที่พลาด {len(wrong_idx)} ไฟล์")
shown = 0
for i in wrong_idx:
    if shown >= 3:
        break
    path, label = test_pairs[i]
    print(f"[{shown + 1}] {label} -> {LABELS[int(y_pred[i])]}")
    display(Audio(load_wave(path), rate=SAMPLES))
    shown += 1


**คำตอบข้อ 2:** คู่แย่สุดมักเป็น `_unknown_` → คำสั่งตัวใดตัวหนึ่ง (หรือกลับกัน) เช่น unk → go / unk → no เหตุผลที่ฟังเจอซ้ำ ๆ: (1) คำในกลุ่ม unknown หลายคำมี "จังหวะ-รูปทรง" คล้ายคำสั่ง (เช่น "bird", "four" สั้นและเสียงสระคล้าย go/no) (2) บางคลิปพูดเบา/เงียบเกือบเป็น silence โมเดลจึงเดา คำสั่งที่พลาดบ่อยสุดของมันเอง (3) ผู้พูด test ที่โมเดลไม่เคยเห็น — นี่คือแบบฝึกหัดเดิมของ M1/M3 แต่เป็นเสียง: การฟังตัวอย่างที่พลาดบอกเหตุผลที่ตัวเลขรวมบอกไม่ได้


## ข้อ 3 — recall ต่อ class: class ไหน "อ่อน" ที่สุด?


In [ ]:
recalls = cm.diagonal() / cm.sum(axis=1)
ranked = sorted(zip(SHORT, recalls), key=lambda p: p[1])
for name, rec in ranked:
    print(f"{name:>6}: {rec:.3f}")


**คำตอบข้อ 3:** จากรอบ seed 42 class ที่ recall ต่ำสุดมักเป็น **`_unknown_`** (recall ~0.8–0.9) ตามด้วย `_silence_` — เหตุผล: `_unknown_` ต้องครอบคลุม 25 คำที่หลากหลายมาก ในขณะที่ข้อมูลเทรนของมันถูก cap ไว้แค่ 800 ตัวอย่าง (และ แต่ละคำในกลุ่มนี้เจอเพียงไม่กี่สิบตัวอย่าง) — โมเดลจึงจำ คำสั่ง 10 คำได้ดีกว่า "จำว่าอะไรไม่ใช่คำสั่ง" ข้อสรุปที่ใช้ได้จริง: ในระบบฟังคำสั่งจริง ให้เพิ่มความหลากหลายของ unknown ในการเทรน และ/หรือใช้ค่าความมั่นใจ (softmax) ต่ำ เป็นเกณฑ์ตอบ "ไม่รู้จัก" แทนการเดาคำสั่ง — ลด false alarm โดยไม่ต้องเทรนเพิ่ม


## ข้อ 4 — คำนวณ WER ด้วยมือ ก่อนเชื่อ jiwer


In [ ]:
import jiwer  # noqa: E402

ref = "เครื่องวัดแสดงค่า 25.4 องศาเซลเซียส"
hyp_a = "เครื่องมือวัดแสดงค่า 25.4 องศาเซลเซียส"
hyp_b = "เครื่องวัดแสดงค่า 25.4"

guess_a = 1 / 3
guess_b = 1 / 3
print("hyp_a เดา:", round(guess_a, 3),
      "| jiwer:", round(jiwer.wer(ref, hyp_a), 3))
print("hyp_b เดา:", round(guess_b, 3),
      "| jiwer:", round(jiwer.wer(ref, hyp_b), 3))


**คำตอบข้อ 4:** ref มี 3 คำ (นับตามช่องว่าง: เครื่องวัดแสดงค่า / 25.4 / องศาเซลเซียส) — hyp_a แทนคำแรก = S=1 → WER = 1/3 = **0.333** hyp_b ตกหล่น "องศาเซลเซียส" = D=1 → WER = 1/3 = **0.333** jiwer ให้ค่าตรงกันทั้งคู่ — ประเด็น: **WER เท่ากันแต่ความผิดคนละแบบ** (แทนคำ อาจยังใช้งานได้ ตกหล่นค่าตัวเลขอาจอันตรายในงานเมโทรโลยี) ตัวเลขเดียวจึงไม่พอ — ต้องดูตัวอย่างจริง + CER ประกอบ (เหมือนบทเรียน M2 เรื่อง CER)


## ข้อ 5 — Demucs กับเพลงของคุณ (demo ที่รันเองได้)

เฉลยใช้โครงเดียวกับแบบฝึกหัด — ถ้าไม่อัปโหลดเพลงจะใช้เพลงตัวอย่างสังเคราะห์ (คำตอบด้านล่างเขียนจากเพลงสังเคราะห์ เพราะโครงสร้างชัดเจน แต่เมื่อฟังเพลงจริง ข้อสรุปเชิงวิเคราะห์เหมือนกัน)


In [ ]:
import glob  # noqa: E402
import librosa  # noqa: E402
import torchaudio.functional as TF  # noqa: E402
from demucs.apply import apply_model  # noqa: E402
from demucs.pretrained import get_model  # noqa: E402

def make_toy_song(seconds=8, sr=44100):
    """เพลงตัวอย่าง: กลอง+เบส+อาร์เปจโจ+โซโล่ (แยกชิ้นง่าย)."""
    t = np.arange(int(sr * seconds)) / sr
    out = np.zeros((2, len(t)), dtype=np.float32)
    rng_song = np.random.default_rng(3)
    for beat in np.arange(0, seconds, 0.5):
        i0 = int(beat * sr)
        n = int(0.05 * sr)
        burst = rng_song.uniform(-1, 1, n)
        env = np.exp(-np.arange(n) / (0.01 * sr))
        out[:, i0:i0 + n] += 0.6 * burst * env
    for step, freq in enumerate([55, 62, 49, 55] * 2):
        i0 = int(step * sr)
        n = int(0.9 * sr)
        tone = np.sin(2 * np.pi * freq * np.arange(n) / sr)
        env = np.minimum(np.arange(n) / (0.05 * sr), 1.0)
        env *= np.exp(-np.arange(n) / (0.7 * sr))
        out[:, i0:i0 + n] += 0.5 * tone * env
    for step in range(int(seconds * 2)):
        freq = [440, 523, 659, 523][step % 4]
        i0 = int(step * 0.5 * sr)
        n = int(0.45 * sr)
        tone = np.sin(2 * np.pi * freq * np.arange(n) / sr)
        env = np.exp(-np.arange(n) / (0.3 * sr))
        out[:, i0:i0 + n] += 0.25 * 0.3 * tone * env
    vib = 220 * (1 + 0.03 * np.sin(2 * np.pi * 5 * t))
    env = 0.4 * (0.5 + 0.5 * np.sin(2 * np.pi * 0.5 * t))
    solo = np.sin(2 * np.pi * np.cumsum(vib) / sr)
    out += 0.35 * env * solo
    peak = np.abs(out).max()
    return (0.9 * out / peak).astype(np.float32)

PATTERNS = [
    "/kaggle/working/*.mp3", "/kaggle/working/*.wav",
    "/kaggle/working/*.m4a", "/kaggle/working/*.flac",
    "/kaggle/input/**/*.mp3", "/kaggle/input/**/*.wav",
]
found = []
for pattern in PATTERNS:
    found += glob.glob(pattern, recursive=True)
song_path = found[0] if found else "/kaggle/working/toy_song.wav"
if song_path.endswith("toy_song.wav") and not found:
    toy = make_toy_song()
    sf.write(song_path, toy.T, 44100)
    print("ไม่มีเพลงอัปโหลด — บันทึกเพลงตัวอย่างแทน")
print("ใช้เพลง:", song_path)


โค้ดแยกเสียง (รันต่อ — ครั้งแรกจะดาวน์โหลดโมเดล ~80 MB):

In [ ]:
demucs_model = get_model("htdemucs")
demucs_model.to(DEVICE).eval()
TARGET_SR = demucs_model.samplerate
audio, sr = librosa.load(song_path, sr=None, mono=False)
if audio.ndim == 1 or audio.shape[0] == 1:
    audio = (
        np.repeat(audio, 2, axis=0)
        if audio.ndim > 1 else np.stack([audio, audio])
    )
elif audio.shape[0] > 2:
    audio = audio[:2]
wav = torch.from_numpy(audio.astype(np.float32))
if sr != TARGET_SR:
    wav = TF.resample(wav, sr, TARGET_SR)
wav = wav[:, :TARGET_SR * 60]
ref = wav.mean(0)
wav_norm = (wav - ref.mean()) / (ref.std() + 1e-8)
with torch.no_grad():
    stems = apply_model(
        demucs_model,
        wav_norm[None].to(DEVICE),
        device=DEVICE,
        split=True,
        overlap=0.25,
        progress=True,
    )[0]
stems = stems * ref.std() + ref.mean()

print("=== mix ===")
display(Audio(wav.cpu().numpy(), rate=TARGET_SR))
for k, name in enumerate(demucs_model.sources):
    print(f"=== stem: {name} ===")
    display(Audio(stems[k].cpu().numpy(), rate=TARGET_SR))


**คำตอบข้อ 5:** กับเพลงสังเคราะห์ — stem ที่สะอาดสุดคือ **drums** (คลิกสั้นคม ๆ ตำแหน่งถูกทุกครึ่งวินาที) และ **bass** (โน้ตต่ำครบทุก 1 วินาที) ส่วน stem vocals ของเพลงสังเคราะห์คือเสียงแกว่ง 220 Hz — htdemucs ไม่เคยเห็นเสียงสังเคราะห์แบบนี้ จึงมักไล่ตามผิด ๆ และหลงเหลือ echo ของอาร์เปจโจใน `other` — สิ่งที่ฟังเจอเป็น artifact ทั่วไปของ source separation: เสียงที่ "รูปแบบไม่คุ้น" มักกระจายผิด stem กับเพลงจริงของผู้เรียน: กลอง/เบสแยกค่อนข้างสะอาด (จังหวะ-ความถี่ชัด) เสียงร้องกับเครื่องดนตรีที่เล่นช่วงเสียงเดียวกันมักปนกัน บทเรียนเดียวกับทุกโมดูล: ต้อง**ฟัง/ดูตัวอย่างจริง**ประกอบตัวเลข ไม่เชื่อ "แยกได้ 100%"


## สรุป (เฉลย)

- โมเดลเล็กลง 4 เท่า: accuracy ต่างกันเพียง ~1–4% — กับโจทย์ที่โครงสร้างชัด โมเดลเล็กก็พอ (คุ้มเวลา/พลังงานเวลา deploy)
- class อ่อนสุด: `_unknown_` — ความหลากหลายของข้อมูลเทรนสำคัญ; ใช้เกณฑ์ความมั่นใจลด false alarm
- WER เท่ากันแต่ผิดคนละแบบ — ต้องดูตัวอย่าง + CER ประกอบ
- Demucs แยกได้จริงแต่มี artifact — โมเดลสำเร็จรูปก็มีขอบเขต ต้องวัดและฟังเองเสมอ
